# AI Programming — Lecture 6
## Lab 1-3: Validation, Overfitting, and Early Stopping — Wine

이번 실습에서는 **Wine Dataset**을 이용하여
train / validation / test set을 구분하고,
학습 과정에서 나타나는 **overfitting**과 **early stopping**을 확인합니다.

### 실습 목표
- 데이터를 train / validation / test set으로 나눕니다.
- validation performance를 이용해 overfitting을 관찰합니다.
- `ModelCheckpoint`로 model을 저장합니다.
- learning curve를 시각화합니다.
- `EarlyStopping`을 이용해 학습을 자동으로 종료합니다.

> 슬라이드의 코드를 가능한 한 그대로 사용합니다.

## 0. 데이터 준비

다음 파일을 Google Drive에 업로드하세요.

```text
My Drive > Colab Notebooks > data > wine.csv
```

모델 저장을 위해 다음 폴더도 준비합니다.

```text
My Drive > Colab Notebooks > models > wine
```

## 1. Dataset Split 및 기본 모델 학습

In [ ]:
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

drive.mount('/content/drive')

df = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/data/wine.csv',
    header=None
)

X = df.iloc[:, 0:12]
y = df.iloc[:, 12]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42,
    stratify=y
)

model = Sequential()
model.add(Input(shape=(12,)))
model.add(Dense(30, activation='relu'))
model.add(Dense(12, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.summary()

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=500,
    validation_split=0.25
)

score = model.evaluate(X_test, y_test)

print(
    'Test loss:',
    score[0],
    'Test accuracy:',
    score[1]
)

### Dataset Split

슬라이드의 설정에서는 최종적으로 다음 비율이 됩니다.

```text
Train       60%
Validation  20%
Test        20%
```

- train set: parameter update
- validation set: training 중 성능 확인
- test set: 최종 평가

## 2. Model 저장하기

In [ ]:
model.save(
    '/content/drive/MyDrive/Colab Notebooks/models/model_temp.keras'
)

저장된 `.keras` 파일에는 model architecture와 trained weights가 함께 저장됩니다.

## 3. ModelCheckpoint 사용하기

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

modelpath = (
    '/content/drive/MyDrive/Colab Notebooks/models/wine/'
    '{epoch:02d}-{val_accuracy:.4f}.keras'
)

checkpointer = ModelCheckpoint(
    filepath=modelpath,
    verbose=1
)

### Complete Code for Checkpoints

In [ ]:
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import ModelCheckpoint

drive.mount('/content/drive')

df = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/data/wine.csv',
    header=None
)

X = df.iloc[:, 0:12]
y = df.iloc[:, 12]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42,
    stratify=y
)

model = Sequential()
model.add(Input(shape=(12,)))
model.add(Dense(30, activation='relu'))
model.add(Dense(12, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

modelpath = (
    '/content/drive/MyDrive/Colab Notebooks/models/wine/'
    '{epoch:02d}-{val_accuracy:.4f}.keras'
)

checkpointer = ModelCheckpoint(
    filepath=modelpath,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=500,
    validation_split=0.25,
    verbose=0,
    callbacks=[checkpointer]
)

score = model.evaluate(X_test, y_test)

print(
    'Test loss:',
    score[0],
    'Test accuracy:',
    score[1]
)

## 4. History Object 확인

In [ ]:
hist_df = pd.DataFrame(history.history)
print(hist_df)

`history.history`에는 epoch별로 다음 값들이 저장됩니다.

```text
loss
accuracy
val_loss
val_accuracy
```

## 5. Overfitting을 관찰하기 위한 긴 학습

슬라이드에서는 overfitting을 명확하게 보기 위해 `epochs=2000`으로 학습합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

drive.mount('/content/drive')

df = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/data/wine.csv',
    header=None
)

X = df.iloc[:, 0:12]
y = df.iloc[:, 12]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42,
    stratify=y
)

model = Sequential()
model.add(Input(shape=(12,)))
model.add(Dense(30, activation='relu'))
model.add(Dense(12, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=2000,
    batch_size=500,
    validation_split=0.25
)

score = model.evaluate(X_test, y_test)

print(
    'Test loss:',
    score[0],
    'Test accuracy:',
    score[1]
)

## 6. Learning Curves 시각화

In [ ]:
hist_df = pd.DataFrame(history.history)

y_val_loss = hist_df['val_loss']
y_train_loss = hist_df['loss']

x_len = np.arange(len(y_train_loss))

plt.plot(
    x_len,
    y_val_loss,
    'o',
    c='red',
    markersize=2,
    label='Validation loss'
)

plt.plot(
    x_len,
    y_train_loss,
    'o',
    c='blue',
    markersize=2,
    label='Train loss'
)

plt.legend(loc='upper right')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.ylim([0.01, 0.17])
plt.show()

y_val_acc = hist_df['val_accuracy']
y_train_acc = hist_df['accuracy']

x_len = np.arange(len(y_train_acc))

plt.plot(
    x_len,
    y_val_acc,
    'o',
    c='red',
    markersize=2,
    label='Validation accuracy'
)

plt.plot(
    x_len,
    y_train_acc,
    'o',
    c='blue',
    markersize=2,
    label='Train accuracy'
)

plt.legend(loc='lower right')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.ylim([0.7, 1.0])
plt.show()

### 확인할 내용

Overfitting이 발생하면 일반적으로

- train loss는 계속 감소
- validation loss는 어느 시점 이후 다시 증가

하는 패턴을 볼 수 있습니다.

Validation loss가 가장 낮았던 시점을 찾는 것이 중요합니다.

## 7. Early Stopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=20
)

`patience=20`은 validation loss가 20 epoch 동안 개선되지 않으면 학습을 종료한다는 의미입니다.

## 8. Best Model 저장 + Early Stopping

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

modelpath = (
    '/content/drive/MyDrive/Colab Notebooks/models/wine/'
    'best_wine_model.keras'
)

checkpointer = ModelCheckpoint(
    filepath=modelpath,
    monitor='val_loss',
    verbose=0,
    save_best_only=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=2000,
    batch_size=500,
    validation_split=0.25,
    verbose=1,
    callbacks=[
        early_stopping_callback,
        checkpointer
    ]
)

## 9. Best Model 불러오기 및 Test Evaluation

In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model(
    '/content/drive/MyDrive/Colab Notebooks/models/wine/'
    'best_wine_model.keras'
)

score = best_model.evaluate(
    X_test,
    y_test
)

print(
    'Test loss:',
    score[0],
    'Test accuracy:',
    score[1]
)

### 핵심 흐름

```text
Train set
→ parameter update

Validation set
→ overfitting 확인
→ best epoch 선택
→ early stopping

Test set
→ 최종 성능 평가
```

## 10. 직접 해보기

1. `patience=5`, `20`, `50`으로 바꾸어 stopping epoch를 비교해 보세요.
2. `epochs=2000`을 유지하되 early stopping을 켠 경우와 끈 경우를 비교해 보세요.
3. train loss와 validation loss가 갈라지기 시작하는 시점을 관찰해 보세요.
4. 마지막 epoch의 model이 아니라 **validation loss가 가장 낮았던 model**을 사용하는 이유를 설명해 보세요.